In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd
import os

In [3]:
data_dir = str(PROJECT_ROOT / "data" / "processed" / "TEP") + "/"
models_dir = str(PROJECT_ROOT / "models" / "TEP") + "/"
ablation_plain_residuals_dir = models_dir + "anomaly_detection_ablation_plain_residuals/"
x_dir = ablation_plain_residuals_dir + "X/"
y_dir = ablation_plain_residuals_dir + "y/"
weights_dir = ablation_plain_residuals_dir + "weights/"
y_pred_dir = ablation_plain_residuals_dir + "y_pred/"
threshold_selection_dir = ablation_plain_residuals_dir + "threshold_selection/"
x_faulty_dir = threshold_selection_dir + "X/"
y_faulty_dir = threshold_selection_dir + "y/"
y_pred_faulty_dir = threshold_selection_dir + "y_pred/"

for directory in [ablation_plain_residuals_dir, x_dir, y_dir, weights_dir, y_pred_dir, threshold_selection_dir, x_faulty_dir, y_faulty_dir, y_pred_faulty_dir]:
    os.makedirs(directory, exist_ok=True)

# 1. Fitting Full VAR Models with Plain Residual Scoring

In [4]:
training_file_name = "TEP_Training.parquet"

In [5]:
df = pd.read_parquet(data_dir + training_file_name)

In [6]:
n_faultfree_samples = df[df["faultNumber"] == 0].shape[0]
df_faultfree = df.iloc[:n_faultfree_samples]
n_samples_per_ts = df[df["faultNumber"] == 0].groupby("simulationRun").size().max()
df_faultfree = df_faultfree.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [7]:
feature_names = df_faultfree.columns
n_features = len(feature_names)
child_parents_pairs = {}

for child_idx in range(n_features):
    parents = []
    for feature_idx in range(n_features):
        if child_idx != feature_idx:
            parents.append(feature_names[feature_idx])
    child_parents_pairs[feature_names[child_idx]] = parents

In [8]:
process_children = ['X6', 'X7', 'X8', 'X9', 'X11', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X22']
n_process_children = len(process_children)
process_child_parents_pairs = {child: child_parents_pairs[child] for child in process_children}

In [9]:
faultfree = df[df["faultNumber"] == 0]

n_faultfree_ts = faultfree["simulationRun"].nunique()

n_samples_per_faultfree_ts = (faultfree.groupby("simulationRun").size().max())

In [10]:
max_lag = 1

In [11]:
for child, parents in child_parents_pairs.items():
    df_X = df_faultfree[parents + [child]]
    y = df_faultfree[child]

    df_X_lagged = pd.concat([df_X.shift(i).add_suffix(f"_lag{i}") for i in range(1, max_lag + 1)], axis=1)

    mask = []
    for ts_start_idx in range(0, n_faultfree_samples, n_samples_per_ts):
        mask.extend(range(ts_start_idx + max_lag, ts_start_idx + n_samples_per_ts))

    df_X_lagged = df_X_lagged.iloc[mask]
    y = y.iloc[mask]

    X = np.ones((df_X_lagged.shape[0], df_X_lagged.shape[1] + 1))
    X[:, 1:] = df_X_lagged.values
    y = y.to_numpy()

    beta, *_ = np.linalg.lstsq(X, y, rcond=None)

    np.save(x_dir + f"X_child_{child}_lag_{max_lag}.npy", X)
    np.save(y_dir + f"y_child_{child}_lag_{max_lag}.npy", y)
    np.save(weights_dir + f"beta_child_{child}_lag_{max_lag}.npy", beta)

In [12]:
for child, parents in child_parents_pairs.items():
    X = np.load(x_dir + f"X_child_{child}_lag_{max_lag}.npy")
    y = np.load(y_dir + f"y_child_{child}_lag_{max_lag}.npy")           
    beta = np.load(weights_dir + f"beta_child_{child}_lag_{max_lag}.npy")
    
    y_pred = X @ beta

    np.save(y_pred_dir + f"y_pred_child_{child}_lag_{max_lag}.npy", y_pred)

# 2. Threshold Selection

In [13]:
training_file_name = "TEP_Training.parquet"

In [14]:
df = pd.read_parquet(data_dir + training_file_name)

In [15]:
n_faulty_samples = df[df["faultNumber"] != 0].shape[0]
df_faulty = df.iloc[-n_faulty_samples:]

n_samples_per_faulty_ts = df[df["faultNumber"] != 0].groupby(["faultNumber", "simulationRun"]).size().max()
n_faulty_ts = df[df["faultNumber"] != 0].groupby(["sample"]).size().max()

df_faulty = df_faulty.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [16]:
faultfree = df[df["faultNumber"] == 0]

n_faultfree_ts = faultfree["simulationRun"].nunique()

n_samples_per_faultfree_ts = (faultfree.groupby("simulationRun").size().max())

In [17]:
n_samples_val = n_faulty_ts * (n_samples_per_faulty_ts - max_lag)
res_val = np.empty((n_samples_val, n_process_children))

for child, parents in process_child_parents_pairs.items():

    df_X_val = df_faulty[parents + [child]]
    y_val = df_faulty[child]

    df_X_val_lagged = pd.concat([df_X_val.shift(i).add_suffix(f"_lag{i}") for i in range(1, max_lag + 1)], axis=1)

    mask = []
    for ts_start_idx in range(0, n_faulty_samples, n_samples_per_faulty_ts):
        mask.extend(range(ts_start_idx + max_lag, ts_start_idx + n_samples_per_faulty_ts))

    df_X_val_lagged = df_X_val_lagged.iloc[mask]
    y_val = y_val.iloc[mask]

    X_val = np.ones((df_X_val_lagged.shape[0], df_X_val_lagged.shape[1] + 1))
    X_val[:, 1:] = df_X_val_lagged.values
    y_val = y_val.to_numpy()

    beta = np.load(weights_dir + f"beta_child_{child}_lag_{max_lag}.npy")

    y_val_pred = X_val @ beta

    res_val[:, process_children.index(child)] = y_val - y_val_pred   

    np.save(x_faulty_dir + f"X_child_{child}.npy", X_val)
    np.save(y_faulty_dir + f"y_child_{child}.npy", y_val)
    np.save(y_pred_faulty_dir + f"y_pred_child_{child}_lag_{max_lag}.npy", y_val_pred)

In [18]:
scores_raw = np.mean(res_val ** 2, axis=1)
scores_2d = scores_raw.reshape((n_faulty_ts, n_samples_per_faulty_ts - max_lag))
df_scores_2d = pd.DataFrame(scores_2d)

In [19]:
n_faultfree_samples_per_ts = 20
faultfree_idxs = range(n_faultfree_samples_per_ts - max_lag)

y_anomaly_2d = np.ones_like(scores_2d, dtype=int)
y_anomaly_2d[:, faultfree_idxs] = 0

In [20]:
min_window = 6
max_window = 6
min_consecutive_alarms = 5
max_consecutive_alarms = 5

windows = range(min_window, max_window + 1)
consecutive_alarms_list = range(min_consecutive_alarms, max_consecutive_alarms + 1)

y_anomaly_2d_red = y_anomaly_2d[:, max_window-1:]
idx_first_anomaly = n_faultfree_samples_per_ts - max_lag - max_window + 1

y_ts_anomaly_2d = np.empty((n_faulty_ts, 2))
y_ts_anomaly_2d[:, 0] = y_anomaly_2d_red[:, :idx_first_anomaly].max(axis=1)
y_ts_anomaly_2d[:, 1] = y_anomaly_2d_red[:, idx_first_anomaly:].max(axis=1)
y_ts_anomaly = y_ts_anomaly_2d.reshape(-1)

df_results = pd.DataFrame(columns=[
    "Window Size",
    "Consecutive Alarms",
    "Threshold",
    "Sample Accuracy",
    "Sample TPR",
    "Sample FPR",
    "Sample F1",
    "Trajectory-Phase Accuracy",
    "Trajectory-Phase TPR",
    "Trajectory-Phase FPR",
    "Trajectory-Phase F1",
    "Median Anomaly Detection Delay"
])

for w in windows:

    scores_2d_smoothed = df_scores_2d.T.rolling(window=w).mean().values.T
    scores_2d_smoothed = scores_2d_smoothed[:, max_window-1:]

    max_scores_smoothed = scores_2d_smoothed[:, :idx_first_anomaly].max(axis=1)
    threshold = np.max(max_scores_smoothed)

    raw_breaches = (scores_2d_smoothed > threshold).astype(int)

    for c in consecutive_alarms_list:
        if c > 1:
            y_anomaly_pred_2d = (
                pd.DataFrame(raw_breaches.T).rolling(window=c, min_periods=c).sum() == c
            ).astype(int).values.T
            
            y_anomaly_pred_2d = np.nan_to_num(y_anomaly_pred_2d, nan=0.0)

            eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)
            eval_mask_2d[:, :c-1] = False
        else:
            y_anomaly_pred_2d = raw_breaches
            eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)

        y_anomaly = y_anomaly_2d_red.reshape(-1)
        y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)
        eval_mask = eval_mask_2d.reshape(-1)

        y_anomaly_eval = y_anomaly[eval_mask]
        y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

        tp = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 1))
        fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
        fn = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 0))
        tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))

        accuracy = (tp + tn) / (tp + fp + fn + tn)
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

        y_ts_anomaly_pred_2d = np.empty((n_faulty_ts, 2))
        y_ts_anomaly_pred_2d[:, 0] = y_anomaly_pred_2d[:, :idx_first_anomaly].max(axis=1)

        faulty_phase_preds = y_anomaly_pred_2d[:, idx_first_anomaly:]
        y_ts_anomaly_pred_2d[:, 1] = faulty_phase_preds.max(axis=1)

        detected_mask = faulty_phase_preds.any(axis=1)
        if np.any(detected_mask):
            idx_first_anomaly_pred = faulty_phase_preds[detected_mask].argmax(axis=1)
            median_idx_first_anomaly_pred = np.median(idx_first_anomaly_pred)
        else:
            median_idx_first_anomaly_pred = np.nan
        
        y_ts_anomaly_pred = y_ts_anomaly_pred_2d.reshape(-1)

        tp_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 1))
        fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
        fn_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 0))
        tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))

        accuracy_ts = (tp_ts + tn_ts) / (tp_ts + fp_ts + fn_ts + tn_ts)
        tpr_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
        fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0
        precision_ts = tp_ts / (tp_ts + fp_ts) if (tp_ts + fp_ts) > 0 else 0
        recall_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
        f1_ts = 2 * precision_ts * recall_ts / (precision_ts + recall_ts) if precision_ts + recall_ts > 0 else 0

        df_result = pd.DataFrame({
            "Window Size": [w],
            "Consecutive Alarms": [c],
            "Threshold": [threshold],
            "Sample Accuracy": [accuracy],
            "Sample TPR": [tpr],
            "Sample FPR": [fpr],
            "Sample F1": [f1],
            "Trajectory-Phase Accuracy": [accuracy_ts],
            "Trajectory-Phase TPR": [tpr_ts],
            "Trajectory-Phase FPR": [fpr_ts],
            "Trajectory-Phase F1": [f1_ts],
            "Median Anomaly Detection Delay": [median_idx_first_anomaly_pred],
        })

        df_results = pd.concat([df_results, df_result], ignore_index=True)

best_idx = df_results["Trajectory-Phase F1"].idxmax()
best_window = df_results['Window Size'][best_idx]
best_cons_alarms = df_results['Consecutive Alarms'][best_idx]
best_threshold = df_results['Threshold'][best_idx]

print("Highest Trajectory-Phase F1:")
print(f"\tWindow Size = {best_window}")
print(f"\tConsecutive Alarms = {best_cons_alarms}")
print(f"\tThreshold = {best_threshold}")

with pd.option_context('display.max_rows', None):
    display(df_results.sort_values("Trajectory-Phase F1", ascending=False))

Highest Trajectory-Phase F1:
	Window Size = 6
	Consecutive Alarms = 5
	Threshold = 0.5043298041769662


<ipykernel>\841345593.py:117: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Median Anomaly Detection Delay
0,6,5,0.50433,0.566001,0.556959,0.0,0.715445,0.9191,0.8382,0.0,0.911979,18.0


# 2. Test on Faulty Time Series

In [21]:
test_file_name = "TEP_Testing.parquet"

In [22]:
df = pd.read_parquet(data_dir + test_file_name)

In [23]:
n_faulty_samples = df[df["faultNumber"] != 0].shape[0]
df_faulty = df.iloc[-n_faulty_samples:]

n_samples_per_faulty_ts = df[df["faultNumber"] != 0].groupby(["faultNumber", "simulationRun"]).size().max()
n_faulty_ts = df[df["faultNumber"] != 0].groupby(["sample"]).size().max()

df_faulty = df_faulty.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [24]:
max_lag = 1

In [25]:
n_samples_test = n_faulty_ts * (n_samples_per_faulty_ts - max_lag)
res_test = np.empty((n_samples_test, n_process_children))

for child, parents in process_child_parents_pairs.items():

    df_X_test = df_faulty[parents + [child]]
    y_test = df_faulty[child]

    df_X_test_lagged = pd.concat([df_X_test.shift(i).add_suffix(f"_lag{i}") for i in range(1, max_lag + 1)], axis=1)

    mask = []
    for ts_start_idx in range(0, n_faulty_samples, n_samples_per_faulty_ts):
        mask.extend(range(ts_start_idx + max_lag, ts_start_idx + n_samples_per_faulty_ts))

    df_X_test_lagged = df_X_test_lagged.iloc[mask]
    y_test = y_test.iloc[mask]

    X_test = np.ones((df_X_test_lagged.shape[0], df_X_test_lagged.shape[1] + 1))
    X_test[:, 1:] = df_X_test_lagged.values
    y_test = y_test.to_numpy()

    beta = np.load(weights_dir + f"beta_child_{child}_lag_{max_lag}.npy")

    y_test_pred = X_test @ beta

    res_test[:, process_children.index(child)] = y_test - y_test_pred   

In [26]:
scores_raw = np.mean(res_test ** 2, axis=1)
scores_2d = scores_raw.reshape((n_faulty_ts, n_samples_per_faulty_ts - max_lag))
df_scores_2d = pd.DataFrame(scores_2d)

In [27]:
n_faultfree_samples_per_ts = 160
faultfree_idxs = range(n_faultfree_samples_per_ts - max_lag)

y_anomaly_2d = np.ones_like(scores_2d, dtype=int)
y_anomaly_2d[:, faultfree_idxs] = 0

In [28]:
window = int(best_window)
consecutive_alarms = int(best_cons_alarms)
threshold = best_threshold

y_anomaly_2d_red = y_anomaly_2d[:, window-1:]
idx_first_anomaly = n_faultfree_samples_per_ts - max_lag - window + 1

y_ts_anomaly_2d = np.empty((n_faulty_ts, 2))
y_ts_anomaly_2d[:, 0] = y_anomaly_2d_red[:, :idx_first_anomaly].max(axis=1)
y_ts_anomaly_2d[:, 1] = y_anomaly_2d_red[:, idx_first_anomaly:].max(axis=1)
y_ts_anomaly = y_ts_anomaly_2d.reshape(-1)

df_results = pd.DataFrame(columns=[
    "Window Size",
    "Consecutive Alarms",
    "Threshold",
    "Sample Accuracy",
    "Sample TPR",
    "Sample FPR",
    "Sample F1",
    "Trajectory-Phase Accuracy",
    "Trajectory-Phase TPR",
    "Trajectory-Phase FPR",
    "Trajectory-Phase F1",
    "Median Anomaly Detection Delay"
])

scores_2d_smoothed = df_scores_2d.T.rolling(window=window).mean().values.T
scores_2d_smoothed = scores_2d_smoothed[:, window-1:]

raw_breaches = (scores_2d_smoothed > threshold).astype(int)

y_anomaly_pred_2d = (
    pd.DataFrame(raw_breaches.T).rolling(
        window=consecutive_alarms, min_periods=consecutive_alarms
    ).sum() == consecutive_alarms
).astype(int).values.T

eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)
eval_mask_2d[:, :consecutive_alarms-1] = False

y_anomaly = y_anomaly_2d_red.reshape(-1)
y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)
eval_mask = eval_mask_2d.reshape(-1)

y_anomaly_eval = y_anomaly[eval_mask]
y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

tp = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 1))
fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
fn = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 0))
tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))

accuracy = (tp + tn) / (tp + fp + fn + tn)
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

y_ts_anomaly_pred_2d = np.empty((n_faulty_ts, 2))
y_ts_anomaly_pred_2d[:, 0] = y_anomaly_pred_2d[:, :idx_first_anomaly].max(axis=1)

faulty_phase_preds = y_anomaly_pred_2d[:, idx_first_anomaly:]
y_ts_anomaly_pred_2d[:, 1] = faulty_phase_preds.max(axis=1)

detected_mask = faulty_phase_preds.any(axis=1)
if np.any(detected_mask):
    idx_first_anomaly_pred = faulty_phase_preds[detected_mask].argmax(axis=1)
    median_idx_first_anomaly_pred = np.median(idx_first_anomaly_pred)
else:
    median_idx_first_anomaly_pred = np.nan

y_ts_anomaly_pred = y_ts_anomaly_pred_2d.reshape(-1)

tp_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 1))
fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
fn_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 0))
tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))

accuracy_ts = (tp_ts + tn_ts) / (tp_ts + fp_ts + fn_ts + tn_ts)
tpr_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0
precision_ts = tp_ts / (tp_ts + fp_ts) if (tp_ts + fp_ts) > 0 else 0
recall_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
f1_ts = 2 * precision_ts * recall_ts / (precision_ts + recall_ts) if precision_ts + recall_ts > 0 else 0

df_result = pd.DataFrame({
    "Window Size": [window],
    "Consecutive Alarms": [consecutive_alarms],
    "Threshold": [threshold],
    "Sample Accuracy": [accuracy],
    "Sample TPR": [tpr],
    "Sample FPR": [fpr],
    "Sample F1": [f1],
    "Trajectory-Phase Accuracy": [accuracy_ts],
    "Trajectory-Phase TPR": [tpr_ts],
    "Trajectory-Phase FPR": [fpr_ts],
    "Trajectory-Phase F1": [f1_ts],
    "Median Anomaly Detection Delay": [median_idx_first_anomaly_pred],
})

df_results = pd.concat([df_results, df_result], ignore_index=True)

with pd.option_context('display.max_rows', None):
    display(df_results)

<ipykernel>\40951338.py:103: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Median Anomaly Detection Delay
0,6,5,0.50433,0.630247,0.560926,0.00004,0.718706,0.92215,0.8483,0.004,0.915942,19.0


In [29]:
from carl.tep_figure_exports import save_tep_figure_metrics
save_tep_figure_metrics(df_results, PROJECT_ROOT, 'Raw residual', 'faulty_test', 'score_ablations')


Saved TEP figure metrics to <repo>\reports\results\TEP\figure_metrics\score_ablations_raw_residual_faulty_test.csv


# 3. Test on Fault-Free Time Series

In [30]:
df = pd.read_parquet(data_dir + test_file_name)

In [31]:
n_faultfree_samples = df[df["faultNumber"] == 0].shape[0]
df_faultfree = df.iloc[:n_faultfree_samples]

n_samples_per_faultfree_ts = df[df["faultNumber"] == 0].groupby(["faultNumber", "simulationRun"]).size().max()
n_faultfree_ts = df[df["faultNumber"] == 0].groupby(["sample"]).size().max()

df_faultfree = df_faultfree.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [32]:
n_samples_test = n_faultfree_ts * (n_samples_per_faultfree_ts - max_lag)
res_test = np.empty((n_samples_test, n_process_children))

for child, parents in process_child_parents_pairs.items():

    df_X_test = df_faultfree[parents + [child]]
    y_test = df_faultfree[child]

    df_X_test_lagged = pd.concat([df_X_test.shift(i).add_suffix(f"_lag{i}") for i in range(1, max_lag + 1)], axis=1)

    mask = []
    for ts_start_idx in range(0, n_faultfree_samples, n_samples_per_faultfree_ts):
        mask.extend(range(ts_start_idx + max_lag, ts_start_idx + n_samples_per_faultfree_ts))

    df_X_test_lagged = df_X_test_lagged.iloc[mask]
    y_test = y_test.iloc[mask]

    X_test = np.ones((df_X_test_lagged.shape[0], df_X_test_lagged.shape[1] + 1))
    X_test[:, 1:] = df_X_test_lagged.values
    y_test = y_test.to_numpy()

    beta = np.load(weights_dir + f"beta_child_{child}_lag_{max_lag}.npy")

    y_test_pred = X_test @ beta

    res_test[:, process_children.index(child)] = y_test - y_test_pred   

In [33]:
scores_raw = np.mean(res_test ** 2, axis=1)
scores_2d = scores_raw.reshape((n_faultfree_ts, n_samples_per_faultfree_ts - max_lag))
df_scores_2d = pd.DataFrame(scores_2d)

In [34]:
y_anomaly_2d = np.zeros_like(scores_2d, dtype=int)

In [35]:
window = int(best_window)
consecutive_alarms = int(best_cons_alarms)
threshold = best_threshold

y_anomaly_2d_red = y_anomaly_2d[:, window-1:]

y_ts_anomaly = y_anomaly_2d_red.max(axis=1)
y_ts_anomaly = y_ts_anomaly.reshape(-1)

df_results = pd.DataFrame(columns=[
    "Window Size",
    "Consecutive Alarms",
    "Threshold",
    "Sample FPR",
    "Trajectory-Phase FPR",
])

scores_2d_smoothed = df_scores_2d.T.rolling(window=window).mean().values.T
scores_2d_smoothed = scores_2d_smoothed[:, window-1:]

raw_breaches = (scores_2d_smoothed > threshold).astype(int)

y_anomaly_pred_2d = (
    pd.DataFrame(raw_breaches.T).rolling(
        window=consecutive_alarms, min_periods=consecutive_alarms
    ).sum() == consecutive_alarms
).astype(int).values.T

eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)
eval_mask_2d[:, :consecutive_alarms-1] = False

y_anomaly = y_anomaly_2d_red.reshape(-1)
y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)
eval_mask = eval_mask_2d.reshape(-1)

y_anomaly_eval = y_anomaly[eval_mask]
y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

y_ts_anomaly_pred = y_anomaly_pred_2d.max(axis=1)
y_ts_anomaly_pred = y_ts_anomaly_pred.reshape(-1)

fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))

fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0

df_result = pd.DataFrame({
    "Window Size": [window],
    "Consecutive Alarms": [consecutive_alarms],
    "Threshold": [threshold],
    "Sample FPR": [fpr],
    "Trajectory-Phase FPR": [fpr_ts],
})

df_results = pd.concat([df_results, df_result], ignore_index=True)

with pd.option_context('display.max_rows', None):
    display(df_results)

<ipykernel>\2516363846.py:60: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


,Window Size,Consecutive Alarms,Threshold,Sample FPR,Trajectory-Phase FPR
0,6,5,0.50433,0.000025,0.018


In [36]:
from carl.tep_figure_exports import save_tep_figure_metrics
save_tep_figure_metrics(df_results, PROJECT_ROOT, 'Raw residual', 'healthy_test', 'score_ablations')


Saved TEP figure metrics to <repo>\reports\results\TEP\figure_metrics\score_ablations_raw_residual_healthy_test.csv
